In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from acoustic_change_detector import AcousticChangeDetector
from phoneme_validator import PhonemeValidator
from hybrid_rnn_hmm_decoder import HybridRNNHMMDecoder

In [2]:
# Define paths
path_bids = './SingleWordProductionDutch-iBIDS'  # Path to the BIDS dataset
path_output = './features'  # Path to save extracted features
path_results = './results'  # Path to save results

In [3]:
# 1. Initialize the custom decoder and detector
from custom_decoder import CustomBrainAudioDecoder
custom_decoder = CustomBrainAudioDecoder(
    path_bids=path_bids,
    path_output=path_output,
    path_results=path_results,
    debug_mode=True
)

CustomBrainAudioDecoder: Initializing CustomBrainAudioDecoder with debug_mode=True
CustomBrainAudioDecoder [DEBUG]: Initialized additional models for comparison


In [4]:
# 2. Load and prepare the data
print("Loading participants and extracting features...")
# This loads metadata for all participants and extracts high gamma features
# from the EEG data, which will be used for model training.
custom_decoder.load_participants()
custom_decoder.extract_features_all_participants()

# This identifies which channels are most relevant for speech decoding
# and will be used to stratify participants by signal quality.
custom_decoder.analyze_channels_across_participants()

Loading participants and extracting features...
Found 10 participants:
  1. sub-01
  2. sub-02
  3. sub-03
  4. sub-04
  5. sub-05
  6. sub-06
  7. sub-07
  8. sub-08
  9. sub-09
  10. sub-10
Extracting features for sub-01...
Extracting features for sub-02...
Extracting features for sub-03...
Extracting features for sub-04...
Extracting features for sub-05...
Extracting features for sub-06...
Extracting features for sub-07...
Extracting features for sub-08...
Extracting features for sub-09...
Extracting features for sub-10...
Analyzing channels across participants...

Analyzing sub-01...
Loading existing results for sub-01...

Analyzing sub-02...
Loading existing results for sub-02...

Analyzing sub-03...
Loading existing results for sub-03...

Analyzing sub-04...
Loading existing results for sub-04...

Analyzing sub-05...
Loading existing results for sub-05...

Analyzing sub-06...
Loading existing results for sub-06...

Analyzing sub-07...
Loading existing results for sub-07...

Analy

{'sub-01': {'LA1': {'correlation': 0.04148753333452151,
   'region': 'Left-Cerebral-White-Matter',
   'index': 0,
   'correlations_by_fold': array([[ 0.08394009,  0.07796735,  0.08899915,  0.10506899,  0.10643324,
            0.11347047,  0.11084663,  0.12216204,  0.0718188 ,  0.10921887,
            0.12796517,  0.11166413,  0.11494964,  0.10910039,  0.11195502,
            0.10776214,  0.10254293,  0.09298878,  0.08279124,  0.06506515,
            0.05911008,  0.06387643,  0.04397557],
          [-0.06357089, -0.0659276 , -0.07813055, -0.07606354, -0.0997465 ,
           -0.11010209, -0.10795719, -0.10517158, -0.10803587, -0.10056936,
           -0.10580493, -0.1035407 , -0.10226087, -0.1071615 , -0.107879  ,
           -0.10602132, -0.10578259, -0.11972009, -0.12518309, -0.12121532,
           -0.12849383, -0.12694488, -0.11717225],
          [ 0.10336479,  0.10583034,  0.10604898,  0.11196844,  0.12030556,
            0.12286169,  0.11416333,  0.10867079,  0.10916221,  0.11888269,


In [5]:
# 3. Stratify participants and create train/test split
print("Creating stratified cross-word split...")
participant_strata = custom_decoder.stratify_participants_by_channel_quality(
    channel_correlation_threshold=0.1 
)
split_result = custom_decoder.create_stratified_cross_word_split(
    participant_strata=participant_strata,
    test_ratio=0.2,
    min_word_freq=1,
    random_seed=42
)

Creating stratified cross-word split...
CustomBrainAudioDecoder [DEBUG]: Stratifying participants based on channel quality...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-01...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-02...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-03...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-04...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-05...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-06...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-07...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-08...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-09...
CustomBrainAudioDecoder [DEBUG]: Analyzing channels for sub-10...
CustomBrainAudioDecoder: 
Participant stratification results:
CustomBrainAudioDecoder:   Participants with most relevant channels: 3
CustomBrainAudioDecoder:   Participants with relevant channels: 4
CustomBrainAudi

CustomBrainAudioDecoder [DEBUG]: Loaded 307521 word markers
CustomBrainAudioDecoder [DEBUG]: First 10 word markers: ['' 'vogeltje' 'vogeltje' 'vogeltje' 'vogeltje' 'vogeltje' 'vogeltje'
 'vogeltje' 'vogeltje' 'vogeltje']
CustomBrainAudioDecoder [DEBUG]: Found 100 word onsets
CustomBrainAudioDecoder [DEBUG]: First 5 word onsets: [(1, 'vogeltje'), (3072, 'wel'), (6147, 'mooi'), (9225, 'daarna'), (12302, 'kasteel')]
CustomBrainAudioDecoder [DEBUG]: Found 100 unique words with at least 1 occurrences
CustomBrainAudioDecoder [DEBUG]: Word 'vogeltje': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'wel': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'mooi': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'daarna': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'kasteel': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word statistics:
CustomBrainAudioDecoder [DEBUG]:   Total unique words: 100
CustomBrainAudioDecoder [DEBUG]:   Total word instances: 100
CustomBrainAudioDecoder [DEBUG]

CustomBrainAudioDecoder [DEBUG]: Loaded 307497 word markers
CustomBrainAudioDecoder [DEBUG]: First 10 word markers: ['' 'verlost' 'verlost' 'verlost' 'verlost' 'verlost' 'verlost' 'verlost'
 'verlost' 'verlost']
CustomBrainAudioDecoder [DEBUG]: Found 100 word onsets
CustomBrainAudioDecoder [DEBUG]: First 5 word onsets: [(1, 'verlost'), (3076, 'terugvinden'), (6152, 'er'), (9225, 'hem'), (12301, 'tot')]
CustomBrainAudioDecoder [DEBUG]: Found 100 unique words with at least 1 occurrences
CustomBrainAudioDecoder [DEBUG]: Word 'verlost': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'terugvinden': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'er': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'hem': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word 'tot': 1 instances
CustomBrainAudioDecoder [DEBUG]: Word statistics:
CustomBrainAudioDecoder [DEBUG]:   Total unique words: 100
CustomBrainAudioDecoder [DEBUG]:   Total word instances: 100
CustomBrainAudioDecoder [DEBUG]: 
Overlap st

In [6]:
# 4. Initialize the acoustic change detector
print("Initializing acoustic change detector...")
detector = AcousticChangeDetector(
    min_segment_duration=0.03,
    max_segment_duration=0.3,
    distance_metric='cosine',
    smoothing_window=3,
    peak_threshold=0.5,
    decoder=custom_decoder,
    debug_mode=True 
)

Initializing acoustic change detector...
AcousticChangeDetector: Initialized with DEBUG_MODE=True


In [7]:
# 5. Initialize validator
validator = PhonemeValidator(detector=detector)
validator.enable_debug()

PhonemeValidator: Initialized with DEBUG_MODE=False
PhonemeValidator: Debug mode enabled


In [8]:
# 6. Pass split_result to detector for data accumulation
detector.split_result = split_result

In [9]:
# 7. Accumulate phoneme data
print("Accumulating training data...")
train_data = detector.accumulate_phoneme_data(
    num_batches=5,
    batch_size=32,
    feature_extraction_method='high_gamma',
    batch_type='train'
)

Accumulating training data...
AcousticChangeDetector: Accumulating train data from 5 batches...
AcousticChangeDetector: Processing batch 1/5
AcousticChangeDetector [DEBUG]: Processing batch with 32 instances
AcousticChangeDetector [DEBUG]: Original batch keys: ['eeg_segments', 'audio_segments', 'words', 'participant_ids', 'metadata', 'spectrogram_segments']
AcousticChangeDetector [DEBUG]: EEG segments in batch: True
AcousticChangeDetector [DEBUG]: Spectrogram segments in batch: True
AcousticChangeDetector [DEBUG]: Processing instance 0/32: 12
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3078, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: 12
AcousticChangeDetector [DEBUG]: Estimated 5 phonemes for '12'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3075, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: mij
AcousticChangeDetector [DEBUG]: Estimated 2 phonemes for 'mij'
AcousticChangeDetector [DEBUG]: Added EEG segme

AcousticChangeDetector [DEBUG]: Processing segment 5 with shape (3076, 54)
AcousticChangeDetector [DEBUG]: Processing segment 6 with shape (3071, 117)
AcousticChangeDetector [DEBUG]: Processing segment 7 with shape (3077, 127)
AcousticChangeDetector [DEBUG]: Processing segment 8 with shape (3075, 127)
AcousticChangeDetector [DEBUG]: Processing segment 9 with shape (3076, 60)
AcousticChangeDetector [DEBUG]: Processing segment 10 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 11 with shape (2647, 122)
AcousticChangeDetector [DEBUG]: Processing segment 12 with shape (3078, 127)
AcousticChangeDetector [DEBUG]: Processing segment 13 with shape (3077, 127)
AcousticChangeDetector [DEBUG]: Processing segment 14 with shape (3078, 122)
AcousticChangeDetector [DEBUG]: Processing segment 15 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 16 with shape (3069, 54)
AcousticChangeDetector [DEBUG]: Processing segment 17 with shape (3072, 127)
Acousti

AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3077, 54)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: sprong
AcousticChangeDetector [DEBUG]: Estimated 5 phonemes for 'sprong'
AcousticChangeDetector [DEBUG]: Processing instance 30/32: naar
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: naar
AcousticChangeDetector [DEBUG]: Estimated 3 phonemes for 'naar'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3081, 54)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: waren
AcousticChangeDetector [DEBUG]: Estimated 5 phonemes for 'waren'
AcousticChangeDetector [DEBUG]: Enhanced batch contains 133 phoneme segments
AcousticChangeDetector [DEBUG]: Found 34 unique phonemes
AcousticChangeDetector [DEBUG]: Preparing phoneme-level training data...
AcousticChangeDetector [DEBUG]: Successfully imported extractHG function
AcousticChangeDetector [DEBUG]: Found

AcousticChangeDetector [DEBUG]: Processing segment 1 with shape (3079, 127)
AcousticChangeDetector [DEBUG]: Processing segment 2 with shape (3079, 122)
AcousticChangeDetector [DEBUG]: Processing segment 3 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 4 with shape (3076, 115)
AcousticChangeDetector [DEBUG]: Processing segment 5 with shape (3081, 54)
AcousticChangeDetector [DEBUG]: Processing segment 6 with shape (3075, 117)
AcousticChangeDetector [DEBUG]: Processing segment 7 with shape (3077, 127)
AcousticChangeDetector [DEBUG]: Processing segment 8 with shape (3078, 127)
AcousticChangeDetector [DEBUG]: Processing segment 9 with shape (3073, 60)
AcousticChangeDetector [DEBUG]: Processing segment 10 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 11 with shape (3077, 127)
AcousticChangeDetector [DEBUG]: Processing segment 12 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 13 with shape (3077, 127)
AcousticCh

AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3078, 122)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: alsof
AcousticChangeDetector [DEBUG]: Estimated 4 phonemes for 'alsof'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3077, 54)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: door
AcousticChangeDetector [DEBUG]: Estimated 3 phonemes for 'door'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3078, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: sprong
AcousticChangeDetector [DEBUG]: Estimated 5 phonemes for 'sprong'
AcousticChangeDetector [DEBUG]: Processing instance 30/32: naar
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: naar
AcousticChangeDetector [DEBUG]: Estimated 3 phonemes for 'naar'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Detect

AcousticChangeDetector [DEBUG]: Processing segment 1 with shape (3073, 127)
AcousticChangeDetector [DEBUG]: Processing segment 2 with shape (3069, 122)
AcousticChangeDetector [DEBUG]: Processing segment 3 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 4 with shape (3077, 115)
AcousticChangeDetector [DEBUG]: Processing segment 5 with shape (3077, 54)
AcousticChangeDetector [DEBUG]: Processing segment 6 with shape (3080, 117)
AcousticChangeDetector [DEBUG]: Processing segment 7 with shape (3072, 127)
AcousticChangeDetector [DEBUG]: Processing segment 8 with shape (3075, 127)
AcousticChangeDetector [DEBUG]: Processing segment 9 with shape (3075, 60)
AcousticChangeDetector [DEBUG]: Processing segment 10 with shape (3074, 54)
AcousticChangeDetector [DEBUG]: Processing segment 11 with shape (2647, 122)
AcousticChangeDetector [DEBUG]: Processing segment 12 with shape (3071, 115)
AcousticChangeDetector [DEBUG]: Processing segment 13 with shape (3076, 127)
AcousticCha

In [10]:
print("Accumulating test data...")
test_data = detector.accumulate_phoneme_data(
    num_batches=3,
    batch_size=32,
    feature_extraction_method='high_gamma',
    batch_type='test'
)

Accumulating test data...
AcousticChangeDetector: Accumulating test data from 3 batches...
AcousticChangeDetector: Processing batch 1/3
AcousticChangeDetector [DEBUG]: Processing batch with 32 instances
AcousticChangeDetector [DEBUG]: Original batch keys: ['eeg_segments', 'audio_segments', 'words', 'participant_ids', 'metadata', 'spectrogram_segments']
AcousticChangeDetector [DEBUG]: EEG segments in batch: True
AcousticChangeDetector [DEBUG]: Spectrogram segments in batch: True
AcousticChangeDetector [DEBUG]: Processing instance 0/32: 3
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: 3
AcousticChangeDetector [DEBUG]: Estimated 3 phonemes for '3'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3080, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: hun
AcousticChangeDetector [DEBUG]: Estimated 3 phonemes for 'hun'
AcousticChangeDetector [DEBUG]: Added EEG segment with 

AcousticChangeDetector [DEBUG]: Processing segment 2 with shape (3079, 122)
AcousticChangeDetector [DEBUG]: Processing segment 3 with shape (3075, 127)
AcousticChangeDetector [DEBUG]: Processing segment 4 with shape (3075, 115)
AcousticChangeDetector [DEBUG]: Processing segment 5 with shape (3079, 54)
AcousticChangeDetector [DEBUG]: Processing segment 6 with shape (3078, 117)
AcousticChangeDetector [DEBUG]: Processing segment 7 with shape (3071, 127)
AcousticChangeDetector [DEBUG]: Processing segment 8 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 9 with shape (3073, 60)
AcousticChangeDetector [DEBUG]: Processing segment 10 with shape (3080, 117)
AcousticChangeDetector [DEBUG]: Processing segment 11 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 12 with shape (3072, 115)
AcousticChangeDetector [DEBUG]: Processing segment 13 with shape (3077, 127)
AcousticChangeDetector [DEBUG]: Processing segment 14 with shape (3076, 127)
AcousticC

AcousticChangeDetector [DEBUG]: Processing segment 1 with shape (3080, 127)
AcousticChangeDetector [DEBUG]: Processing segment 2 with shape (3076, 122)
AcousticChangeDetector [DEBUG]: Processing segment 3 with shape (3074, 127)
AcousticChangeDetector [DEBUG]: Processing segment 4 with shape (3077, 115)
AcousticChangeDetector [DEBUG]: Processing segment 5 with shape (3078, 54)
AcousticChangeDetector [DEBUG]: Processing segment 6 with shape (3077, 117)
AcousticChangeDetector [DEBUG]: Processing segment 7 with shape (3079, 127)
AcousticChangeDetector [DEBUG]: Processing segment 8 with shape (3073, 127)
AcousticChangeDetector [DEBUG]: Processing segment 9 with shape (3076, 60)
AcousticChangeDetector [DEBUG]: Processing segment 10 with shape (3080, 117)
AcousticChangeDetector [DEBUG]: Processing segment 11 with shape (3076, 127)
AcousticChangeDetector [DEBUG]: Processing segment 12 with shape (3072, 115)
AcousticChangeDetector [DEBUG]: Processing segment 13 with shape (3077, 127)
AcousticCh

AcousticChangeDetector [DEBUG]: Processing instance 30/32: zo
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3074, 127)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: zo
AcousticChangeDetector [DEBUG]: Estimated 2 phonemes for 'zo'
AcousticChangeDetector [DEBUG]: Added EEG segment with shape (3073, 117)
AcousticChangeDetector [DEBUG]: Detecting boundaries for word: 1
AcousticChangeDetector [DEBUG]: Estimated 1 phonemes for '1'
AcousticChangeDetector [DEBUG]: No transcription for word '1'
AcousticChangeDetector [DEBUG]: Enhanced batch contains 135 phoneme segments
AcousticChangeDetector [DEBUG]: Found 28 unique phonemes
AcousticChangeDetector [DEBUG]: Preparing phoneme-level training data...
AcousticChangeDetector [DEBUG]: Successfully imported extractHG function
AcousticChangeDetector [DEBUG]: Found 32 valid phoneme segments out of 32
AcousticChangeDetector [DEBUG]: Processing segment 0 with shape (3074, 127)
AcousticChangeDetector [DEBUG]: Processing seg

In [11]:
# 8. Resolve unknown phonemes for better data quality
# For training data
train_converted = {
    'phoneme_labels': train_data['phoneme_labels'],
    'phoneme_spectrogram_segments': train_data.get('spectrograms', []),
    'phoneme_words': train_data['phoneme_words'],
    'phoneme_positions': train_data.get('phoneme_positions', [0] * len(train_data['phoneme_labels'])),
    'phoneme_participant_ids': train_data.get('phoneme_participant_ids', ['unknown'] * len(train_data['phoneme_labels']))
}

In [12]:
print(f"Training data before resolution: {train_converted['phoneme_labels'].count('?')} unknown phonemes")
resolved_train = validator.resolve_unknown_phonemes(train_converted)
print(f"Training data after resolution: {resolved_train['phoneme_labels'].count('?')} unknown phonemes")
# Update the training data with resolved phonemes
train_data['phoneme_labels'] = resolved_train['phoneme_labels']

Training data before resolution: 59 unknown phonemes
PhonemeValidator [DEBUG]: Attempting to resolve 59 unknown phonemes
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'tot' as 't'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'tot' as 't'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'tot' as 't'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'maantje' as 'm'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'maantje' as 'm'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'maantje' as 'm'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'maantje' as 'm'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'wel' as 'w'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'wel' as 'w'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'wel' as 'w'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'of' as 'ɔf'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 

In [13]:
# For test data
test_converted = {
    'phoneme_labels': test_data['phoneme_labels'],
    'phoneme_spectrogram_segments': test_data.get('spectrograms', []),
    'phoneme_words': test_data['phoneme_words'],
    'phoneme_positions': test_data.get('phoneme_positions', [0] * len(test_data['phoneme_labels'])),
    'phoneme_participant_ids': test_data.get('phoneme_participant_ids', ['unknown'] * len(test_data['phoneme_labels']))}

In [14]:
print(f"Test data before resolution: {test_converted['phoneme_labels'].count('?')} unknown phonemes")
resolved_test = validator.resolve_unknown_phonemes(test_converted)
print(f"Test data after resolution: {resolved_test['phoneme_labels'].count('?')} unknown phonemes")

Test data before resolution: 30 unknown phonemes
PhonemeValidator [DEBUG]: Attempting to resolve 30 unknown phonemes
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'dichtbij' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'dichtbij' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'dichtbij' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'dichtbij' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'dichtbij' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'aan' as 'a'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'aan' as 'a'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'aan' as 'a'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'direct' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'direct' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at position 0 in 'direct' as 'd'
PhonemeValidator [DEBUG]: Resolved phoneme at 

In [15]:
# Update the test data with resolved phonemes
test_data['phoneme_labels'] = resolved_test['phoneme_labels']

In [16]:
# 9. Initialize the Hybrid RNN-HMM Decoder
print("Initializing HybridRNNHMMDecoder")
hybrid_model = HybridRNNHMMDecoder(
    output_dir=os.path.join(path_results, 'hybrid_model'),
    debug_mode=True,
    phonetic_dict=detector.phonetic_dict,
    hmm_states_per_phoneme=3
)

Initializing HybridRNNHMMDecoder
HybridRNNHMMDecoder: Initialized HybridRNNHMMDecoder with hmm_states_per_phoneme=3


In [17]:
# 10. Train the hybrid model
print("Training hybrid model...")
results = hybrid_model.train_with_accumulated_data(
    train_data=train_data,
    test_data=test_data,
    validation_split=0.2,
    nn_epochs=50,
    hmm_iter=50,
    patience=10
)

Training hybrid model...
HybridRNNHMMDecoder: Processing accumulated phoneme data
HybridRNNHMMDecoder: Training hybrid RNN-HMM model
HybridRNNHMMDecoder: Training neural network with 160 samples
HybridRNNHMMDecoder: Maximum sequence length: 295
HybridRNNHMMDecoder: Found 30 unique phonemes: ['a', 'b', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'øk', 'œy', 'ɑ', 'ɔ', 'ɔf', 'ə', 'ɛ', 'ɛi', 'ɪ', 'ʏ']
HybridRNNHMMDecoder: Building neural network with input_shape=(295, 50), output_dim=30


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)        │ (None, 295, 256)            │         183,296 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 295, 256)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 128)                 │         164,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 30)                  │           3,870 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 368,542 (1.41 MB)

 Trainable params: 368,286 (1.40 MB)

...

HybridRNNHMMDecoder: Successfully trained HMM for phoneme 's'
HybridRNNHMMDecoder: Training HMM for phoneme 'ɔf' with 8 samples


Model is not converging.  Current: 493.5373925647276 is not greater than 603.742305912764. Delta is -110.20491334803637
Fitting a model with 188 free scalar parameters with only 60 data points will result in a degenerate solution.


HybridRNNHMMDecoder: Successfully trained HMM for phoneme 'ɔf'
HybridRNNHMMDecoder: Training HMM for phoneme 'ɔ' with 1 samples
HybridRNNHMMDecoder: Warning: Skipping phoneme 'ɔ' with only 1 samples
HybridRNNHMMDecoder: Training HMM for phoneme 'ɪ' with 2 samples
HybridRNNHMMDecoder: Error training HMM for phoneme 'ɪ': n_samples=2 should be >= n_clusters=3.
HybridRNNHMMDecoder: Training HMM for phoneme 'b' with 1 samples
HybridRNNHMMDecoder: Warning: Skipping phoneme 'b' with only 1 samples
HybridRNNHMMDecoder: Training HMM for phoneme 'e' with 1 samples
HybridRNNHMMDecoder: Warning: Skipping phoneme 'e' with only 1 samples
HybridRNNHMMDecoder: Training HMM for phoneme 'p' with 1 samples
HybridRNNHMMDecoder: Warning: Skipping phoneme 'p' with only 1 samples
HybridRNNHMMDecoder: Training HMM for phoneme 'øk' with 1 samples
HybridRNNHMMDecoder: Warning: Skipping phoneme 'øk' with only 1 samples
HybridRNNHMMDecoder: Learning phoneme transition probabilities
HybridRNNHMMDecoder: Processing

C:\Users\irina\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\irina\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\irina\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\irina\anaconda3\Lib\site-packages\sklearn\metrics\_cla

HybridRNNHMMDecoder: Neural network accuracy: 0.0581
HybridRNNHMMDecoder: Hybrid model accuracy: 0.0116
HybridRNNHMMDecoder: Saving hybrid model
HybridRNNHMMDecoder: Model saved successfully


In [18]:
# 11. Print results summary
print("\n===== Results Summary =====")
if results['eval_results'] is not None:
    nn_accuracy = results['eval_results']['nn_accuracy']
    hybrid_accuracy = results['eval_results']['hybrid_accuracy']
    print(f"Neural network accuracy: {nn_accuracy:.4f}")
    print(f"Hybrid model accuracy: {hybrid_accuracy:.4f}")
    
    # Improvement from the HMM component
    improvement = hybrid_accuracy - nn_accuracy
    print(f"Improvement from HMM: {improvement:.4f} ({improvement*100:.1f}%)")

print("\nModel training and evaluation complete!")


===== Results Summary =====
Neural network accuracy: 0.0581
Hybrid model accuracy: 0.0116
Improvement from HMM: -0.0465 (-4.7%)

Model training and evaluation complete!


In [19]:
# 12. Additional evaluation and analysis
if results['eval_results'] is not None:
    # Compare predicted phonemes with true phonemes for a sample
    true_phonemes = results['eval_results']['hybrid_phonemes'][:10]
    pred_phonemes = results['eval_results']['nn_phonemes'][:10]
    
    print("\nSample Predictions:")
    print("True vs Predicted Phonemes")
    print("-----------------------")
    for true, pred in zip(true_phonemes, pred_phonemes):
        print(f"{true:<5} -> {pred:<5} {'✓' if true == pred else '✗'}")
    
    # Print performance by phoneme category
    if results['eval_results']['classification_report'] is not None:
        report = results['eval_results']['classification_report']
        
        # Group phonemes into categories
        vowels = [p for p in hybrid_model.phoneme_list if p in 'aeiouəɑɛɪɔʊʌœyɵ' or p in ['ɛi', 'œy', 'ɑu']]
        consonants = [p for p in hybrid_model.phoneme_list if p not in vowels]
        
        print("\nPerformance by Phoneme Category:")
        print("Vowels:")
        vowel_f1 = 0
        vowel_count = 0
        for v in vowels:
            if v in report:
                vowel_f1 += report[v]['f1-score']
                vowel_count += 1
                print(f"  {v}: F1={report[v]['f1-score']:.4f}, Support={report[v]['support']}")
        
        if vowel_count > 0:
            print(f"  Average vowel F1: {vowel_f1/vowel_count:.4f}")
        
        print("\nConsonants:")
        consonant_f1 = 0
        consonant_count = 0
        for c in consonants:
            if c in report:
                consonant_f1 += report[c]['f1-score']
                consonant_count += 1
                print(f"  {c}: F1={report[c]['f1-score']:.4f}, Support={report[c]['support']}")
        
        if consonant_count > 0:
            print(f"  Average consonant F1: {consonant_f1/consonant_count:.4f}")


Sample Predictions:
True vs Predicted Phonemes
-----------------------
œy    -> ʏ     ✗
t     -> b     ✗
w     -> œy    ✗
ɪ     -> p     ✗
n     -> n     ✓
t     -> e     ✗
w     -> l     ✗
ɪ     -> n     ✗
n     -> œy    ✗
t     -> r     ✗

Performance by Phoneme Category:
Vowels:
  a: F1=0.2000, Support=7
  e: F1=0.3333, Support=3
  i: F1=0.0000, Support=1
  o: F1=0.0000, Support=2
  u: F1=0.0000, Support=0
  œy: F1=0.0000, Support=0
  ɑ: F1=0.2500, Support=5
  ɔ: F1=0.0000, Support=1
  ə: F1=0.0000, Support=8
  ɛ: F1=0.0000, Support=2
  ɛi: F1=0.0000, Support=0
  ɪ: F1=0.0000, Support=0
  Average vowel F1: 0.0653

Consonants:
  b: F1=0.0000, Support=1
  d: F1=0.0000, Support=18
  f: F1=0.0000, Support=0
  h: F1=0.0000, Support=3
  k: F1=0.0000, Support=3
  l: F1=0.2353, Support=3
  m: F1=0.0000, Support=1
  n: F1=0.0000, Support=6
  p: F1=0.0000, Support=0
  r: F1=0.0000, Support=2
  s: F1=0.0000, Support=0
  t: F1=0.0000, Support=2
  v: F1=0.0000, Support=2
  w: F1=0.0000, Support